# 03 — Refined KPIs

- **KPI 1**: Patron de demanda temporal (franja horaria x dia de semana)
- **KPI 2**: Eficiencia economica por zona (revenue/milla, velocidad, top 10)
- **KPI 3**: Impacto de calidad de datos (% descartados por regla, efecto en ingresos)

In [ ]:
%run ../config/pipeline_config

In [ ]:
import logging
import time
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("refined_kpis")

start_time = time.time()
metrics = {"stage": "03_refined_kpis"}

In [ ]:
# Leer tabla trusted
df = spark.table(TRUSTED_CLEAN_TABLE)
total_trusted = df.count()
logger.info(f"Registros leidos de trusted: {total_trusted:,}")

---
## KPI 1 — Demanda Temporal

Viajes, duracion promedio y tarifa por franja horaria (6) y dia de la semana.
Las combinaciones en el percentil 80 se marcan como pico.

In [ ]:
# Agregar franja horaria
df_kpi1 = df.withColumn("hour", F.hour("tpep_pickup_datetime"))

df_kpi1 = df_kpi1.withColumn(
    "time_slot",
    F.when((F.col("hour") >= 0) & (F.col("hour") < 4), F.lit("Madrugada (00-04h)"))
     .when((F.col("hour") >= 4) & (F.col("hour") < 8), F.lit("Manana Temprana (04-08h)"))
     .when((F.col("hour") >= 8) & (F.col("hour") < 12), F.lit("Manana (08-12h)"))
     .when((F.col("hour") >= 12) & (F.col("hour") < 16), F.lit("Tarde (12-16h)"))
     .when((F.col("hour") >= 16) & (F.col("hour") < 20), F.lit("Tarde-Noche (16-20h)"))
     .otherwise(F.lit("Noche (20-24h)"))
)

# Agregar dia de la semana
df_kpi1 = df_kpi1.withColumn("day_of_week_num", F.dayofweek("tpep_pickup_datetime"))
df_kpi1 = df_kpi1.withColumn(
    "day_of_week",
    F.when(F.col("day_of_week_num") == 1, "Domingo")
     .when(F.col("day_of_week_num") == 2, "Lunes")
     .when(F.col("day_of_week_num") == 3, "Martes")
     .when(F.col("day_of_week_num") == 4, "Miercoles")
     .when(F.col("day_of_week_num") == 5, "Jueves")
     .when(F.col("day_of_week_num") == 6, "Viernes")
     .otherwise("Sabado")
)

In [ ]:
# Agregar por franja horaria y dia de la semana
kpi1 = df_kpi1.groupBy("time_slot", "day_of_week", "day_of_week_num").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
    F.round(F.avg("total_amount"), 2).alias("avg_total_amount")
)

# Identificar picos (percentil 80)
threshold = kpi1.approxQuantile("trip_count", [0.8], 0.01)[0]
kpi1 = kpi1.withColumn("is_peak", F.col("trip_count") >= threshold)

# Ordenar para presentacion
kpi1 = kpi1.orderBy("day_of_week_num", "time_slot")

logger.info(f"KPI 1 calculado: {kpi1.count()} combinaciones franja/dia. Umbral de pico: {threshold:,.0f} viajes.")
kpi1.show(42, truncate=False)

In [ ]:
try:
    kpi1.write.format("delta").mode("overwrite").saveAsTable(REFINED_KPI_DEMAND)
    spark.sql(f"COMMENT ON TABLE {REFINED_KPI_DEMAND} IS 'KPI 1: Demanda por franja horaria y dia de semana.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["refined"].items()])
    spark.sql(f"ALTER TABLE {REFINED_KPI_DEMAND} SET TBLPROPERTIES ({props})")
    spark.sql(f"OPTIMIZE {REFINED_KPI_DEMAND} ZORDER BY (time_slot, day_of_week)")

    logger.info(f"'{REFINED_KPI_DEMAND}' escrita.")
except Exception as e:
    logger.error(f"Error escribiendo KPI 1: {e}")
    raise

---
## KPI 2 — Eficiencia Economica por Zona

Revenue por milla, velocidad promedio y ranking. Se filtran viajes con distance > 0.1 mi y duracion > 1 min para evitar divisiones por casi-cero.

In [ ]:
# Filtrar para calculos de eficiencia (evitar division por cero/casi-cero)
df_eff = df.filter(
    (F.col("trip_distance") > 0.1) &
    (F.col("trip_duration_minutes") > 1)
)

# Calcular metricas de eficiencia
df_eff = df_eff.withColumn(
    "revenue_per_mile",
    F.col("total_amount") / F.col("trip_distance")
).withColumn(
    "speed_mph",
    F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
)

# Agregar por zona y borough
kpi2_all = df_eff.filter(
    F.col("pickup_zone").isNotNull()
).groupBy("pickup_borough", "pickup_zone").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("revenue_per_mile"), 2).alias("avg_revenue_per_mile"),
    F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue")
)

# Ranking por ingreso por milla
w = Window.orderBy(F.col("avg_revenue_per_mile").desc())
kpi2_all = kpi2_all.withColumn("rank_by_revenue_per_mile", F.row_number().over(w))

# Top 10
kpi2_top10 = kpi2_all.filter(F.col("rank_by_revenue_per_mile") <= 10)

logger.info(f"KPI 2 calculado: {kpi2_all.count()} zonas analizadas.")
print("\n--- Top 10 Zonas Mas Rentables ---")
kpi2_top10.show(10, truncate=False)

In [ ]:
try:
    kpi2_all.write.format("delta").mode("overwrite").saveAsTable(REFINED_KPI_EFFICIENCY)
    spark.sql(f"COMMENT ON TABLE {REFINED_KPI_EFFICIENCY} IS 'KPI 2: Eficiencia por zona. Filtrar rank_by_revenue_per_mile <= 10 para top 10.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["refined"].items()])
    spark.sql(f"ALTER TABLE {REFINED_KPI_EFFICIENCY} SET TBLPROPERTIES ({props})")
    spark.sql(f"OPTIMIZE {REFINED_KPI_EFFICIENCY} ZORDER BY (pickup_borough)")

    logger.info(f"'{REFINED_KPI_EFFICIENCY}' escrita.")
except Exception as e:
    logger.error(f"Error escribiendo KPI 2: {e}")
    raise

---
## KPI 3 — Impacto de Calidad de Datos

% de registros descartados por regla y su efecto en ingresos totales.

In [ ]:
# Leer registros rechazados
df_rejected = spark.table(TRUSTED_REJECTED_TABLE)
total_raw = spark.table(RAW_TAXI_TABLE).count()

# Ingresos en trusted (sin rechazados)
trusted_revenue = df.agg(F.sum("total_amount")).collect()[0][0] or 0
# Ingresos de registros rechazados
rejected_revenue = df_rejected.agg(F.sum("total_amount")).collect()[0][0] or 0
# Ingresos totales raw (clean + rejected)
total_revenue_raw = trusted_revenue + rejected_revenue

logger.info(f"Ingresos trusted: ${trusted_revenue:,.2f}")
logger.info(f"Ingresos rechazados: ${rejected_revenue:,.2f}")
logger.info(f"Ingresos raw totales: ${total_revenue_raw:,.2f}")

In [ ]:
# Detalle por regla de rechazo
kpi3 = df_rejected.groupBy("rejection_reason").agg(
    F.count("*").alias("rejected_count"),
    F.round(F.sum("total_amount"), 2).alias("estimated_revenue_lost")
).withColumn(
    "rejected_pct",
    F.round(F.col("rejected_count") / F.lit(total_raw) * 100, 2)
).withColumn(
    "revenue_impact_pct",
    F.round(
        F.when(F.lit(total_revenue_raw) != 0, F.col("estimated_revenue_lost") / F.lit(total_revenue_raw) * 100)
         .otherwise(0),
        2
    )
).orderBy(F.col("rejected_count").desc())

print("\n--- Impacto de Calidad por Regla ---")
kpi3.show(truncate=False)

In [ ]:
try:
    kpi3.write.format("delta").mode("overwrite").saveAsTable(REFINED_KPI_QUALITY_IMPACT)
    spark.sql(f"COMMENT ON TABLE {REFINED_KPI_QUALITY_IMPACT} IS 'KPI 3: Registros descartados por regla y efecto en ingresos.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["refined"].items()])
    spark.sql(f"ALTER TABLE {REFINED_KPI_QUALITY_IMPACT} SET TBLPROPERTIES ({props})")

    logger.info(f"'{REFINED_KPI_QUALITY_IMPACT}' escrita.")
except Exception as e:
    logger.error(f"Error escribiendo KPI 3: {e}")
    raise

## Resumen

In [ ]:
elapsed = round(time.time() - start_time, 2)
metrics["duration_seconds"] = elapsed
metrics["kpis_generated"] = 3
metrics["status"] = "SUCCESS"

logger.info(f"Refined KPIs completado en {elapsed}s.")
logger.info(f"KPI 1: {kpi1.count()} filas (demanda temporal)")
logger.info(f"KPI 2: {kpi2_all.count()} zonas (eficiencia economica)")
logger.info(f"KPI 3: {kpi3.count()} reglas (impacto calidad)")

print("\n" + "="*60)
print(json.dumps(metrics, indent=2, default=str))
print("="*60)

try:
    dbutils.notebook.exit(json.dumps(metrics, default=str))
except NameError:
    pass